# Predict, Ship, Compete
## From SQL to Live A/B Test

**Your mission**: Build a model that decides which ads to show to which users to **maximize revenue** (not just clicks).

Your model will be deployed in a live A/B test against other teams. The team that generates the most revenue per impression wins.

---

**Server URL** (set this to the instructor's server):

In [ ]:
SERVER = "http://localhost:8000"  # Change this to the instructor's IP/hostname
TEAM_NAME = "your-team-name"       # Pick a unique team name

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cloudpickle
import time
import io
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
def query(sql, limit=50000):
    """Run a SQL query against the workshop database."""
    resp = requests.post(f"{SERVER}/api/sql", json={"query": sql, "limit": limit})
    resp.raise_for_status()
    data = resp.json()
    return pd.DataFrame(data["rows"], columns=data["columns"])

### Register your team

In [ ]:
resp = requests.post(f"{SERVER}/api/teams/{TEAM_NAME}/register",
                     json={"members": ["Alice", "Bob"]})  # Put your names here
print(resp.json())

---

# Phase 1: Explore the Data (45 min)

You have access to an e-commerce ad database with four tables:
- **users** — 10,000 users with demographics and behavior
- **ads** — 200 ad creatives with product and creative metadata
- **impressions** — 500,000 ad impressions (did the user click?)
- **conversions** — purchases that followed a click

You can also explore interactively at the **SQL Explorer**: `{SERVER}/sql`

## 1.1 Get the lay of the land

In [ ]:
# Check what we're working with
schema = requests.get(f"{SERVER}/api/schema").json()
for table, info in schema["tables"].items():
    cols = [c["name"] for c in info["columns"]]
    print(f"\n{table} ({info['row_count']:,} rows):")
    print(f"  Columns: {', '.join(cols)}")

In [ ]:
# Overall funnel metrics
query("""
SELECT
  COUNT(*) as impressions,
  SUM(clicked) as clicks,
  ROUND(AVG(clicked) * 100, 2) as ctr_pct,
  (SELECT COUNT(*) FROM conversions) as conversions,
  (SELECT ROUND(SUM(revenue), 2) FROM conversions) as total_revenue
FROM impressions
""")

## 1.2 Investigate: Are all clicks equally valuable?

**Key question**: If you optimize purely for CTR (click-through rate), will you also maximize revenue?

In [ ]:
# CTR and revenue by user segments
# Try different groupings: loyalty_tier, device_type, age_group, etc.

segment_stats = query("""
SELECT
  u.loyalty_tier,
  u.device_type,
  COUNT(*) as impressions,
  SUM(i.clicked) as clicks,
  ROUND(AVG(i.clicked) * 100, 2) as ctr_pct,
  COUNT(c.conversion_id) as conversions,
  ROUND(SUM(c.revenue), 2) as total_revenue,
  ROUND(SUM(c.revenue) / COUNT(*), 4) as revenue_per_impression
FROM impressions i
JOIN users u ON i.user_id = u.user_id
LEFT JOIN conversions c ON i.impression_id = c.impression_id
GROUP BY u.loyalty_tier, u.device_type
ORDER BY revenue_per_impression DESC
""")
segment_stats

In [ ]:
# YOUR EXPLORATION: What patterns do you see?
# - Which segments have high CTR but low revenue?
# - Which segments have low CTR but high revenue per impression?
# - What does this mean for your modeling strategy?
#
# Write your own queries below:



In [ ]:
# Do clickbait-y ads generate more revenue?
clickbait_analysis = query("""
SELECT
  CASE
    WHEN a.headline_clickbait_score >= 7 THEN 'high_clickbait'
    WHEN a.headline_clickbait_score >= 4 THEN 'medium_clickbait'
    ELSE 'low_clickbait'
  END as clickbait_level,
  CASE
    WHEN a.creative_quality_score >= 7 THEN 'high_quality'
    WHEN a.creative_quality_score >= 4 THEN 'medium_quality'
    ELSE 'low_quality'
  END as quality_level,
  COUNT(*) as impressions,
  SUM(i.clicked) as clicks,
  ROUND(AVG(i.clicked) * 100, 2) as ctr_pct,
  COUNT(c.conversion_id) as conversions,
  ROUND(SUM(c.revenue), 2) as total_revenue,
  ROUND(SUM(c.revenue) / COUNT(*), 4) as revenue_per_impression
FROM impressions i
JOIN ads a ON i.ad_id = a.ad_id
LEFT JOIN conversions c ON i.impression_id = c.impression_id
GROUP BY clickbait_level, quality_level
ORDER BY revenue_per_impression DESC
""")
clickbait_analysis

### Insight check

Before moving on, make sure you can answer:
1. Do high-CTR user segments also have the highest revenue per impression?
2. Do high-clickbait ads generate more revenue than high-quality ads?
3. What is the conversion rate (given click) for different user types?

**The answer to these questions should inform what your model optimizes for.**

---

# Phase 2: Build Your Model (75 min)

## 2.1 Pull training data

Pull the full dataset with user, ad, and context features joined together.

In [ ]:
# Pull a training dataset with all features
# NOTE: We pull in batches because the API has a row limit

def pull_training_data():
    """Pull the full joined dataset in chunks."""
    batch_size = 50000
    all_data = []
    offset = 0

    while True:
        df = query(f"""
        SELECT
            i.impression_id,
            -- User features
            u.age_group, u.gender, u.device_type, u.region,
            u.account_age_days, u.past_purchases, u.avg_order_value,
            u.sessions_per_week, u.loyalty_tier,
            -- Ad features
            a.category, a.ad_format, a.product_price, a.discount_pct,
            a.creative_quality_score, a.headline_clickbait_score,
            a.brand_familiarity,
            -- Context features
            i.page_type, i.position, i.hour_of_day, i.day_of_week,
            i.session_depth,
            -- Targets
            i.clicked,
            CASE WHEN c.conversion_id IS NOT NULL THEN 1 ELSE 0 END as converted,
            COALESCE(c.revenue, 0) as revenue
        FROM impressions i
        JOIN users u ON i.user_id = u.user_id
        JOIN ads a ON i.ad_id = a.ad_id
        LEFT JOIN conversions c ON i.impression_id = c.impression_id
        ORDER BY i.impression_id
        LIMIT {batch_size} OFFSET {offset}
        """, limit=batch_size)

        if len(df) == 0:
            break
        all_data.append(df)
        offset += batch_size
        print(f"  Pulled {offset:,} rows...")

    return pd.concat(all_data, ignore_index=True)

print("Pulling training data...")
df = pull_training_data()
print(f"\nTotal: {len(df):,} rows")
df.head()

In [ ]:
# Quick sanity check
print(f"CTR: {df['clicked'].mean():.2%}")
print(f"CVR (overall): {df['converted'].mean():.4%}")
print(f"CVR (given click): {df.loc[df['clicked']==1, 'converted'].mean():.2%}")
print(f"Total revenue: ${df['revenue'].sum():,.2f}")
print(f"Revenue per impression: ${df['revenue'].mean():.4f}")

## 2.2 Feature engineering

Prepare features for your model. The model will need to accept a DataFrame with these columns during inference.

In [ ]:
def prepare_features(df):
    """
    Prepare features for modeling.
    Returns a DataFrame with all features ready for sklearn.

    IMPORTANT: This function will also be used at inference time,
    so it must work on a single row or a batch.
    """
    features = pd.get_dummies(
        df[[
            # User features
            'age_group', 'gender', 'device_type', 'region',
            'account_age_days', 'past_purchases', 'avg_order_value',
            'sessions_per_week', 'loyalty_tier',
            # Ad features
            'category', 'ad_format', 'product_price', 'discount_pct',
            'creative_quality_score', 'headline_clickbait_score',
            'brand_familiarity',
            # Context features
            'page_type', 'position', 'hour_of_day', 'day_of_week',
            'session_depth',
        ]],
        columns=['age_group', 'gender', 'device_type', 'region',
                 'loyalty_tier', 'category', 'ad_format', 'page_type'],
        drop_first=True,
        dtype=int,
    )
    return features

X = prepare_features(df)
print(f"Feature matrix: {X.shape}")
print(f"Columns: {list(X.columns)}")

## 2.3 Choose your target

This is the most important decision. What should your model predict?

| Target | Pros | Cons |
|--------|------|------|
| `clicked` (CTR model) | Simple, lots of positive examples | Doesn't capture conversion value |
| `converted` (conversion model) | Captures purchase intent | Very sparse, hard to train |
| `revenue` (revenue model) | Directly optimizes the metric | Very sparse, noisy |
| **Expected value** (composite) | Best alignment with objective | Requires thoughtful construction |

Think about it: the A/B test scores you on **revenue per impression**. What target best approximates this?

$$\text{Expected revenue} = P(\text{click}) \times P(\text{convert} | \text{click}) \times E[\text{revenue} | \text{convert}]$$

In [ ]:
# Option A: Simple CTR model
y_click = df['clicked'].values

# Option B: Direct revenue model
y_revenue = df['revenue'].values

# Option C: Build a composite target (think about this!)
# Hint: can you combine multiple models or create a smarter target?

In [ ]:
# Split data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_click, test_size=0.2, random_state=42  # Change target as needed
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2.4 Train your model

Start simple and iterate. Remember: your model needs to be fast at inference time (Phase 3).

In [ ]:
# Example: Logistic Regression (fast, interpretable baseline)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict_proba(X_test)[:, 1]
print(f"Logistic Regression:")
print(f"  AUC: {roc_auc_score(y_test, y_pred_lr):.4f}")
print(f"  Log Loss: {log_loss(y_test, y_pred_lr):.4f}")

In [ ]:
# Example: LightGBM (more powerful, still pretty fast)
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    num_leaves=31,
    verbose=-1,
)
lgb_model.fit(X_train, y_train)

y_pred_lgb = lgb_model.predict_proba(X_test)[:, 1]
print(f"LightGBM:")
print(f"  AUC: {roc_auc_score(y_test, y_pred_lgb):.4f}")
print(f"  Log Loss: {log_loss(y_test, y_pred_lgb):.4f}")

In [ ]:
# YOUR TURN: Try different models, features, targets, hyperparameters
# Ideas:
#   - Random Forest, XGBoost, neural network
#   - Add interaction features (e.g., user_aov * product_price)
#   - Train separate CTR and CVR models, combine them
#   - Use regression on revenue instead of classification
#   - Weight samples by revenue potential



## 2.5 Evaluate on the right metric

AUC measures ranking quality. But we care about revenue. Let's evaluate properly.

In [ ]:
# Simulate the A/B test scoring locally
# For each impression, the model scores all candidate ads.
# The ad with the highest score gets shown.
# Revenue is determined by the ground truth (clicked + converted + revenue columns).

# Simple evaluation: for each impression in the test set,
# what's the correlation between model scores and actual revenue?

test_df = df.iloc[X_test.index].copy()
test_df['model_score'] = y_pred_lgb  # or whichever model you're evaluating

# How well does the model rank high-revenue impressions?
top_10pct = test_df.nlargest(int(len(test_df) * 0.1), 'model_score')
bottom_90pct = test_df.nsmallest(int(len(test_df) * 0.9), 'model_score')

print(f"Revenue per impression (top 10% by model score): ${top_10pct['revenue'].mean():.4f}")
print(f"Revenue per impression (bottom 90%):              ${bottom_90pct['revenue'].mean():.4f}")
print(f"Revenue per impression (overall test set):        ${test_df['revenue'].mean():.4f}")
print(f"\nLift from targeting top 10%: {top_10pct['revenue'].mean() / test_df['revenue'].mean():.1f}x")

---

# Phase 3: Optimize for Production (30 min)

Your model must score ads in real-time. The simulation has a **latency budget** (default: 50ms).
If your model is slower, some of your traffic gets a random ad choice instead — penalizing slow models.

## 3.1 Benchmark your model's latency

In [ ]:
def benchmark_latency(model, X_sample, n_trials=100):
    """Measure inference latency for a batch of 10 rows (simulates one request)."""
    batch = X_sample.head(10)  # The simulation scores 10 candidate ads per request
    latencies = []

    for _ in range(n_trials):
        start = time.perf_counter()
        if hasattr(model, 'predict_proba'):
            model.predict_proba(batch)
        else:
            model.predict(batch)
        elapsed_ms = (time.perf_counter() - start) * 1000
        latencies.append(elapsed_ms)

    latencies = np.array(latencies)
    print(f"Latency (batch of 10):")
    print(f"  Median: {np.median(latencies):.2f} ms")
    print(f"  P95:    {np.percentile(latencies, 95):.2f} ms")
    print(f"  P99:    {np.percentile(latencies, 99):.2f} ms")
    print(f"  Max:    {np.max(latencies):.2f} ms")

    budget = 50  # ms
    violations = (latencies > budget).mean()
    print(f"\n  Budget violations (>{budget}ms): {violations:.1%}")
    if violations > 0.1:
        print(f"  WARNING: >10% of requests will be penalized!")
    return latencies

print("Logistic Regression:")
benchmark_latency(lr, X_test)
print("\nLightGBM:")
benchmark_latency(lgb_model, X_test)

In [ ]:
# If your model is too slow, try:
#   - Fewer trees / smaller ensemble
#   - Fewer features (drop low-importance ones)
#   - Simpler model (logistic regression is very fast)
#   - Quantize or compress
#
# Feature importance (for feature selection):
if hasattr(lgb_model, 'feature_importances_'):
    importance = pd.Series(
        lgb_model.feature_importances_,
        index=X.columns
    ).sort_values(ascending=True)
    importance.tail(20).plot.barh(figsize=(10, 6), color='#6c63ff')
    plt.title('Top 20 Feature Importances')
    plt.tight_layout()
    plt.show()

## 3.2 Wrap your model for deployment

The simulator calls `model.predict(features_df)` where `features_df` has the same columns as your training data.
The prediction should be a **score** — higher = better ad to show. This could be:
- P(click) from a CTR model
- P(click) * P(convert|click) from two models combined
- Predicted revenue
- Any scoring function you design

In [ ]:
# Option A: Use a single model directly
# If your model has predict_proba, the simulator will use predict() directly.
# So for classifiers, you may want to wrap them.

class ScoringModel:
    """Wraps one or more models into a scoring function."""

    def __init__(self, model, use_proba=True):
        self.model = model
        self.use_proba = use_proba

    def predict(self, X):
        if self.use_proba and hasattr(self.model, 'predict_proba'):
            return self.model.predict_proba(X)[:, 1]
        return self.model.predict(X)


# Option B: Combine CTR and CVR models
class CompositeModel:
    """Combine CTR model, CVR model, and price info for expected value scoring."""

    def __init__(self, ctr_model, cvr_model):
        self.ctr_model = ctr_model
        self.cvr_model = cvr_model

    def predict(self, X):
        p_click = self.ctr_model.predict_proba(X)[:, 1]
        p_convert = self.cvr_model.predict_proba(X)[:, 1]
        # Expected revenue ~ p(click) * p(convert|click) * price
        price = X['product_price'].values if 'product_price' in X.columns else 1.0
        discount = X['discount_pct'].values if 'discount_pct' in X.columns else 0.0
        effective_price = price * (1 - discount / 100)
        return p_click * p_convert * effective_price


# Example: wrap the LightGBM CTR model
final_model = ScoringModel(lgb_model, use_proba=True)

# Verify it works
test_scores = final_model.predict(X_test.head(10))
print(f"Sample scores: {test_scores}")

In [ ]:
# Final latency check
print("Final model latency:")
benchmark_latency(final_model, X_test)

---

# Phase 4: Deploy & Compete! (45 min)

## 4.1 Save and upload your model

In [ ]:
from pathlib import Path

# Save model using cloudpickle (captures class definitions for the server)
model_path = f"model_{TEAM_NAME}.pkl"
with open(model_path, 'wb') as f:
    cloudpickle.dump(final_model, f)

print(f"Model saved to {model_path}")
print(f"Size: {Path(model_path).stat().st_size / 1024:.1f} KB")

In [ ]:
from pathlib import Path

# Upload to the server
with open(model_path, 'rb') as f:
    resp = requests.post(
        f"{SERVER}/api/teams/{TEAM_NAME}/model",
        files={"model": (model_path, f, "application/octet-stream")}
    )

result = resp.json()
print(f"Upload result: {result}")
if resp.ok:
    print(f"\nModel uploaded successfully!")
    print(f"  Type: {result.get('model_type')}")
    print(f"  Validation prediction: {result.get('validation_prediction'):.4f}")
    print(f"  Validation latency: {result.get('validation_latency_ms'):.2f} ms")
else:
    print(f"\nERROR: {result}")

## 4.2 Watch the live dashboard

Open the dashboard in your browser to watch the A/B test in real time:

**Dashboard URL**: `{SERVER}/dashboard`

The instructor will start the simulation once all teams have uploaded their models.

## 4.3 Iterate!

You can re-upload your model at any time. The simulation will start using your new model immediately.
Your metrics will reset when you upload a new model.

In [ ]:
# Check the leaderboard from here
resp = requests.get(f"{SERVER}/api/leaderboard")
lb = resp.json()

print(f"Simulation running: {lb['running']}")
print(f"Total requests: {lb['total_requests']:,}")
print(f"\nLeaderboard:")
for i, team in enumerate(lb['leaderboard']):
    print(f"  #{i+1} {team['team']:<20} "
          f"Rev/Impr: ${team['revenue_per_impression']:.4f}  "
          f"Revenue: ${team['revenue']:.2f}  "
          f"CTR: {team['ctr']:.2%}  "
          f"Latency: {team['avg_latency_ms']:.1f}ms")

---

# Bonus Challenges

If you finish early, try these:

1. **Multi-objective model**: Train separate P(click) and P(convert|click) models, combine into expected revenue
2. **Feature engineering**: Create interaction features, ratio features, binned features
3. **Calibration**: Use Platt scaling or isotonic regression so P(click) predictions are well-calibrated
4. **Latency optimization**: Can you match LightGBM's accuracy with fewer trees? Try `max_depth=3, n_estimators=50`
5. **Exploration-exploitation**: Should you always pick the highest-scoring ad, or sometimes explore?
6. **Causal thinking**: The historical data shows correlations, but the simulation is causal. What biases might exist in training data from observational ad impressions?

In [ ]:
# Space for bonus work
